## Base Functions

In [ ]:
"""
LOS Prediction Pipeline for NY SPARCS Hospital Inpatient Data
=============================================================
Goal: Predict Length of Stay (integer days) at admission using:
  - Demographics: Age Group, Gender, Race, Ethnicity
  - Diagnosis: CCS Diagnosis Code, APR DRG Code
  - Admission: Type of Admission, Health Service Area, Zip Code (3-digit)
  - Severity: APR Severity of Illness, APR Risk of Mortality

Approach:
  1. Load & clean data
  2. Feature engineering (ICD/CCS grouping, severity encoding)
  3. Train/val/test split (80/10/10)
  4. ROUND 1 — Train full LightGBM model on all features
  5. SHAP feature selection — drop low-contribution features
  6. ROUND 2 — Retrain on pruned feature set
  7. Compare Round 1 vs Round 2 metrics to confirm pruning is safe
  8. Quantile regression on final feature set for prediction intervals
  9. Evaluation: MAE, RMSE, MdAE, calibration plot, residuals, SHAP
"""

import numpy as np
import joblib
import pandas as pd
import lightgbm as lgb
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_absolute_error, mean_squared_error
import matplotlib.pyplot as plt
import shap
import warnings
warnings.filterwarnings("ignore")


# ─────────────────────────────────────────────
# 1. LOAD DATA
# ─────────────────────────────────────────────

def load_sparcs(filepath: str) -> pd.DataFrame:
    """
    Load SPARCS CSV. Adjust column names if your version differs slightly.
    The dataset is ~2M rows; chunked reading is used for safety.
    """
    print("Loading data...")
    df = pd.read_csv(
        filepath,
        dtype=str,           # read everything as string first for safety
        low_memory=False
    )
    print(f"  Loaded {len(df):,} rows, {df.shape[1]} columns")
    return df


# ─────────────────────────────────────────────
# 2. CLEAN & PREPARE
# ─────────────────────────────────────────────

# Canonical column name map — adjust if your file uses different names
COLUMN_MAP = {
    "Length of Stay":                   "los",
    "Age Group":                        "age_group",
    "Gender":                           "gender",
    "Race":                             "race",
    "Ethnicity":                        "ethnicity",
    "Type of Admission":                "admission_type",
    "CCS Diagnosis Code":               "ccs_dx",
    "CCS Procedure Code":               "ccs_proc",
    "APR DRG Code":                     "apr_drg",
    "APR Severity of Illness Code":     "apr_severity",
    "APR Risk of Mortality":            "apr_rom",
    "APR Medical Surgical Description": "med_surg",
    "Health Service Area":              "health_service_area",
    "Zip Code - 3 digits":              "zip3",
    "Patient Disposition":              "disposition",      # leakage risk — see note
}

# APR Severity / ROM label encoding (ordinal)
SEVERITY_MAP = {"1": 1, "2": 2, "3": 3, "4": 4, "Minor": 1, "Moderate": 2, "Major": 3, "Extreme": 4}
ROM_MAP      = {"1": 1, "2": 2, "3": 3, "4": 4, "Minor": 1, "Moderate": 2, "Major": 3, "Extreme": 4}


def clean(df: pd.DataFrame) -> pd.DataFrame:
    print("Cleaning...")

    # Rename to standard names
    df = df.rename(columns={k: v for k, v in COLUMN_MAP.items() if k in df.columns})

    # ── Target: LOS ──────────────────────────────────────────────────────────
    # "120+" is capped in SPARCS — treat as 120
    df["los"] = df["los"].str.replace("+", "", regex=False)
    df["los"] = pd.to_numeric(df["los"], errors="coerce")
    df = df.dropna(subset=["los"])
    df["los"] = df["los"].clip(lower=1).astype(int)   # floor at 1 day

    # Log-transform for modelling
    df["log_los"] = np.log(df["los"])

    # ── Severity / ROM ────────────────────────────────────────────────────────
    if "apr_severity" in df.columns:
        df["apr_severity"] = df["apr_severity"].map(SEVERITY_MAP)
    if "apr_rom" in df.columns:
        df["apr_rom"] = df["apr_rom"].map(ROM_MAP)

    # ── Numeric codes ─────────────────────────────────────────────────────────
    for col in ["ccs_dx", "ccs_proc", "apr_drg"]:
        if col in df.columns:
            df[col] = pd.to_numeric(df[col], errors="coerce")

    # ── Drop leaky columns ────────────────────────────────────────────────────
    # Patient Disposition is known AFTER discharge → leakage if used as input
    # Remove it unless you explicitly want a discharge-time model
    leaky = ["disposition", "Total Charges", "Total Costs"]
    df = df.drop(columns=[c for c in leaky if c in df.columns])

    print(f"  Clean shape: {df.shape}")
    print(f"  LOS stats: median={df['los'].median():.0f}, mean={df['los'].mean():.1f}, "
          f"p95={df['los'].quantile(0.95):.0f}, max={df['los'].max()}")
    return df


# ─────────────────────────────────────────────
# 3. FEATURE ENGINEERING
# ─────────────────────────────────────────────

CATEGORICAL_COLS = [
    "age_group", "gender", "race", "ethnicity",
    "admission_type", "med_surg", "health_service_area", "zip3",
    "ccs_dx", "ccs_proc", "apr_drg",
]

NUMERIC_COLS = [
    "apr_severity", "apr_rom",
]


def engineer_features(df: pd.DataFrame) -> tuple[pd.DataFrame, list]:
    """
    Encode categoricals and return feature matrix + feature list.
    LightGBM handles NaNs natively, so minimal imputation needed.
    """
    print("Engineering features...")

    feature_cols = []

    # Categorical → integer codes (LightGBM handles these as categoricals natively)
    for col in CATEGORICAL_COLS:
        if col in df.columns:
            df[col] = df[col].fillna("Unknown").astype("category")
            feature_cols.append(col)

    for col in NUMERIC_COLS:
        if col in df.columns:
            feature_cols.append(col)

    print(f"  Features used: {feature_cols}")
    return df, feature_cols


# ─────────────────────────────────────────────
# 4. TRAIN / VAL / TEST SPLIT
# ─────────────────────────────────────────────

def split_data(df: pd.DataFrame, feature_cols: list):
    """
    80/10/10 split.
    For a single-year dataset (SPARCS 2015) we do random split.
    If you stack multiple years, switch to a time-based split instead.
    """
    print("Splitting data...")

    X = df[feature_cols]
    y = df["log_los"]
    y_raw = df["los"]

    X_temp, X_test, y_temp, y_test, y_raw_temp, y_raw_test = train_test_split(
        X, y, y_raw, test_size=0.10, random_state=42
    )
    X_train, X_val, y_train, y_val, y_raw_train, y_raw_val = train_test_split(
        X_temp, y_temp, y_raw_temp, test_size=0.111, random_state=42  # 0.111 of 90% ≈ 10%
    )

    print(f"  Train: {len(X_train):,} | Val: {len(X_val):,} | Test: {len(X_test):,}")
    return (X_train, X_val, X_test,
            y_train, y_val, y_test,
            y_raw_train, y_raw_val, y_raw_test)


# ─────────────────────────────────────────────
# 5. MODELS
# ─────────────────────────────────────────────

LGBM_BASE_PARAMS = dict(
    objective="regression",
    metric="rmse",
    n_estimators=2000,
    learning_rate=0.05,
    num_leaves=127,
    min_child_samples=50,
    subsample=0.8,
    colsample_bytree=0.8,
    reg_alpha=0.1,
    reg_lambda=1.0,
    n_jobs=-1,
    random_state=42,
    verbose=-1,

    # GPU settings
    device="gpu",
    gpu_platform_id=0,
    gpu_device_id=0,
)


def train_median_model(X_train, y_train, X_val, y_val, cat_cols):
    """Main model: predict median log(LOS)."""
    print("Training median model...")
    model = lgb.LGBMRegressor(**LGBM_BASE_PARAMS)
    model.fit(
        X_train, y_train,
        eval_set=[(X_val, y_val)],
        categorical_feature=cat_cols,
        callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(200)],
    )
    print(f"  Best iteration: {model.best_iteration_}")
    return model


def train_quantile_models(X_train, y_train, X_val, y_val, cat_cols,
                           quantiles=(0.10, 0.25, 0.75, 0.90)):
    """Quantile models for prediction intervals."""
    print("Training quantile models...")
    q_models = {}
    for q in quantiles:
        params = {**LGBM_BASE_PARAMS,
                  "objective": "quantile",
                  "alpha": q,
                  "metric": "quantile"}
        m = lgb.LGBMRegressor(**params)
        m.fit(
            X_train, y_train,
            eval_set=[(X_val, y_val)],
            categorical_feature=cat_cols,
            callbacks=[lgb.early_stopping(50, verbose=False), lgb.log_evaluation(999)],
        )
        q_models[q] = m
        print(f"  Q{int(q*100)}: done (iter={m.best_iteration_})")
    return q_models


# ─────────────────────────────────────────────
# 6. EVALUATION
# ─────────────────────────────────────────────

def evaluate(model, q_models, X_test, y_raw_test, feature_cols):
    print("\n── Evaluation ──────────────────────────────────")

    # Point predictions (back-transform from log scale)
    log_preds = model.predict(X_test)
    preds = np.exp(log_preds)
    preds_rounded = np.round(preds).astype(int).clip(min=1)

    y_true = y_raw_test.values

    mae  = mean_absolute_error(y_true, preds)
    rmse = np.sqrt(mean_squared_error(y_true, preds))
    mdae = np.median(np.abs(y_true - preds))  # median absolute error (robust)

    print(f"  MAE:  {mae:.2f} days")
    print(f"  RMSE: {rmse:.2f} days")
    print(f"  MdAE: {mdae:.2f} days")

    # Quantile coverage
    if q_models:
        lo = np.exp(q_models[0.10].predict(X_test))
        hi = np.exp(q_models[0.90].predict(X_test))
        coverage_80 = np.mean((y_true >= lo) & (y_true <= hi))
        avg_width   = np.mean(hi - lo)
        print(f"  80% PI coverage: {coverage_80:.1%} (target: 80%)")
        print(f"  Avg PI width:    {avg_width:.2f} days")

    # Calibration plot: predicted vs actual mean LOS by decile
    deciles = pd.qcut(preds, q=10, labels=False, duplicates="drop")
    cal_df = pd.DataFrame({"pred": preds, "actual": y_true, "decile": deciles})
    cal_agg = cal_df.groupby("decile").agg(
        pred_mean=("pred", "mean"),
        actual_mean=("actual", "mean"),
        n=("pred", "count")
    ).reset_index()

    fig, axes = plt.subplots(1, 2, figsize=(14, 5))

    # Calibration
    ax = axes[0]
    ax.plot(cal_agg["pred_mean"], cal_agg["actual_mean"], "o-", color="#2563eb", label="Decile means")
    lims = [min(cal_agg["pred_mean"].min(), cal_agg["actual_mean"].min()),
            max(cal_agg["pred_mean"].max(), cal_agg["actual_mean"].max())]
    ax.plot(lims, lims, "--", color="gray", alpha=0.6, label="Perfect calibration")
    ax.set_xlabel("Predicted mean LOS (days)")
    ax.set_ylabel("Actual mean LOS (days)")
    ax.set_title("Calibration Plot (by prediction decile)")
    ax.legend()
    ax.grid(alpha=0.3)

    # Residual distribution
    ax = axes[1]
    residuals = y_true - preds
    ax.hist(residuals, bins=80, color="#2563eb", alpha=0.7, edgecolor="white")
    ax.axvline(0, color="gray", linestyle="--")
    ax.set_xlabel("Residual (actual − predicted) days")
    ax.set_ylabel("Count")
    ax.set_title("Residual Distribution")
    ax.grid(alpha=0.3)

    plt.tight_layout()
    plt.savefig("/content/drive/MyDrive/LOS/calibration_residuals.png", dpi=150)
    plt.close()
    print("  Saved: calibration_residuals.png")

    return preds, preds_rounded


def plot_residuals(
    model,
    X_test: pd.DataFrame,
    y_raw_test: pd.Series,
    feature_cols: list,
    error_threshold: float = None,
    top_n_errors: int = 20,
    sample_n: int = 50_000,
) -> pd.DataFrame:
    """
    Four-panel residual diagnostic plot to identify large-error data points.

    Panels
    ------
    1. Predicted vs Actual — scatter with perfect-fit line and large errors
       highlighted in red.
    2. Residual vs Predicted — reveals heteroscedasticity (errors growing
       with predicted LOS, common in right-skewed targets).
    3. Absolute error distribution — histogram with the large-error threshold
       marked as a vertical line.
    4. Top-N largest errors — horizontal bar chart of the worst individual
       predictions, showing actual vs predicted days.

    Parameters
    ----------
    model           : trained LightGBM model (Round 1 or Round 2)
    X_test          : test feature matrix
    y_raw_test      : true LOS in days (original scale, not log)
    feature_cols    : list of feature names used by the model
    error_threshold : absolute error (days) above which a point is flagged
                      as a large error. Defaults to the 95th percentile of
                      absolute errors if not specified.
    top_n_errors    : number of worst predictions to show in panel 4.
    sample_n        : max rows to plot in scatter panels (for speed at 2M rows).

    Returns
    -------
    error_df : DataFrame of all test predictions with columns
               [actual, predicted, residual, abs_error, is_large_error]
               — use this to inspect or join back to your original data.
    """
    print("\n── Residual Diagnostic Plot ────────────────────")

    # ── Predictions ──────────────────────────────────────────────────────────
    preds  = np.exp(model.predict(X_test))
    y_true = y_raw_test.values

    residuals = y_true - preds          # positive = under-predicted
    abs_error = np.abs(residuals)

    if error_threshold is None:
        error_threshold = np.percentile(abs_error, 95)
        print(f"  error_threshold not set — using 95th percentile: "
              f"{error_threshold:.1f} days")
    else:
        print(f"  error_threshold: {error_threshold:.1f} days")

    is_large = abs_error >= error_threshold
    pct_large = is_large.mean() * 100
    print(f"  Large-error points: {is_large.sum():,} / {len(y_true):,} "
          f"({pct_large:.1f}%)")

    error_df = pd.DataFrame({
        "actual":         y_true,
        "predicted":      preds,
        "residual":       residuals,
        "abs_error":      abs_error,
        "is_large_error": is_large,
    }, index=X_test.index)

    # ── Sample for scatter panels (plotting 2M points is slow) ───────────────
    if len(error_df) > sample_n:
        # Always include all large-error points; sample the rest
        large_idx  = error_df[error_df["is_large_error"]].index
        normal_idx = error_df[~error_df["is_large_error"]].sample(
            sample_n - len(large_idx), random_state=42
        ).index
        plot_df = error_df.loc[large_idx.union(normal_idx)]
    else:
        plot_df = error_df

    # ── Masks ─────────────────────────────────────────────────────────────────
    mask_n = ~plot_df["is_large_error"]
    mask_l =  plot_df["is_large_error"]
    alpha_normal, alpha_large = 0.15, 0.6

    fig, axes = plt.subplots(2, 2, figsize=(15, 12))
    fig.suptitle(
        f"Residual Diagnostics  |  error threshold = {error_threshold:.1f} days  "
        f"|  large errors = {pct_large:.1f}% of test set",
        fontsize=13, y=1.01,
    )

    # ── Panel 1: Predicted vs Actual ─────────────────────────────────────────
    ax = axes[0, 0]
    ax.scatter(plot_df.loc[mask_n, "predicted"], plot_df.loc[mask_n, "actual"],
               c="#2563eb", alpha=alpha_normal, s=6, linewidths=0, label="Normal")
    ax.scatter(plot_df.loc[mask_l, "predicted"], plot_df.loc[mask_l, "actual"],
               c="#ef4444", alpha=alpha_large, s=14, linewidths=0,
               label=f"Large error (≥{error_threshold:.0f} days)")
    lim = max(plot_df["predicted"].max(), plot_df["actual"].max())
    ax.plot([0, lim], [0, lim], "--", color="gray", linewidth=1, alpha=0.7,
            label="Perfect fit")
    ax.set_xlabel("Predicted LOS (days)")
    ax.set_ylabel("Actual LOS (days)")
    ax.set_title("Predicted vs Actual")
    ax.legend(fontsize=8, markerscale=2)
    ax.grid(alpha=0.2)

    # ── Panel 2: Residual vs Predicted ───────────────────────────────────────
    ax = axes[0, 1]
    ax.scatter(plot_df.loc[mask_n, "predicted"], plot_df.loc[mask_n, "residual"],
               c="#2563eb", alpha=alpha_normal, s=6, linewidths=0)
    ax.scatter(plot_df.loc[mask_l, "predicted"], plot_df.loc[mask_l, "residual"],
               c="#ef4444", alpha=alpha_large, s=14, linewidths=0,
               label=f"Large error (≥{error_threshold:.0f} days)")
    ax.axhline(0, color="gray", linestyle="--", linewidth=1)
    ax.axhline( error_threshold, color="#ef4444", linestyle=":", linewidth=1, alpha=0.6)
    ax.axhline(-error_threshold, color="#ef4444", linestyle=":", linewidth=1, alpha=0.6)
    ax.set_xlabel("Predicted LOS (days)")
    ax.set_ylabel("Residual  (actual − predicted)")
    ax.set_title("Residual vs Predicted\n(funnel shape = errors grow with LOS — expected)")
    ax.legend(fontsize=8, markerscale=2)
    ax.grid(alpha=0.2)

    # ── Panel 3: Absolute error distribution ─────────────────────────────────
    ax = axes[1, 0]
    ax.hist(abs_error, bins=100, color="#2563eb", alpha=0.75, edgecolor="white")
    ax.axvline(error_threshold, color="#ef4444", linestyle="--", linewidth=1.5,
               label=f"Threshold: {error_threshold:.1f} days")
    ax.axvline(abs_error.mean(), color="#f59e0b", linestyle="--", linewidth=1.2,
               label=f"Mean AE: {abs_error.mean():.1f} days")
    ax.set_xlabel("Absolute error (days)")
    ax.set_ylabel("Count")
    ax.set_title("Absolute Error Distribution")
    ax.legend(fontsize=8)
    ax.grid(alpha=0.2)

    # ── Panel 4: Top-N largest errors ────────────────────────────────────────
    ax = axes[1, 1]
    top = (
        error_df.nlargest(top_n_errors, "abs_error")
        .reset_index(drop=True)
        .iloc[::-1]   # flip so largest is at top
    )
    y_pos = np.arange(len(top))
    ax.barh(y_pos, top["actual"],    height=0.4, color="#94a3b8", label="Actual")
    ax.barh(y_pos, top["predicted"], height=0.4, color="#2563eb", alpha=0.8,
            label="Predicted", left=0)
    for i, (_, row) in enumerate(top.iterrows()):
        ax.text(max(row["actual"], row["predicted"]) + 0.3, i,
                f"  err={row['residual']:+.0f}d", va="center", fontsize=7,
                color="#ef4444" if row["abs_error"] >= error_threshold else "#374151")
    ax.set_yticks(y_pos)
    ax.set_yticklabels([f"#{i+1}" for i in range(len(top))], fontsize=7)
    ax.set_xlabel("LOS (days)")
    ax.set_title(f"Top {top_n_errors} Largest Errors")
    ax.legend(fontsize=8)
    ax.grid(axis="x", alpha=0.2)

    plt.tight_layout()
    plt.savefig("/content/drive/MyDrive/LOS/residual_diagnostics.png", dpi=150,
                bbox_inches="tight")
    plt.close()
    print("  Saved: residual_diagnostics.png")

    # ── Summary of large-error cases ─────────────────────────────────────────
    print(f"\n  Large-error summary (abs_error ≥ {error_threshold:.1f} days):")
    print(f"    Median actual LOS in large-error cases: "
          f"{error_df.loc[is_large, 'actual'].median():.0f} days")
    print(f"    % that are under-predictions (residual > 0): "
          f"{(error_df.loc[is_large, 'residual'] > 0).mean():.0%}")
    print(f"    % that are over-predictions  (residual < 0): "
          f"{(error_df.loc[is_large, 'residual'] < 0).mean():.0%}")

    return error_df


def explain(model, X_test, feature_cols, n_samples=5000):
    """SHAP feature importance."""
    print("\n── SHAP Explainability ─────────────────────────")
    sample = X_test.sample(min(n_samples, len(X_test)), random_state=42)
    explainer = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(sample)

    plt.figure(figsize=(10, 7))
    shap.summary_plot(shap_values, sample, feature_names=feature_cols,
                      show=False, max_display=15)
    plt.tight_layout()
    plt.savefig("/content/drive/MyDrive/LOS/shap_summary.png", dpi=150)
    plt.close()
    print("  Saved: shap_summary.png")


# ─────────────────────────────────────────────
# 7. SHAP FEATURE SELECTION
# ─────────────────────────────────────────────

def compute_shap_importance(
    model,
    X_train: pd.DataFrame,
    feature_cols: list,
    n_samples: int = 10_000,
) -> pd.DataFrame:
    """
    Compute mean |SHAP| for each feature and save a bar chart for inspection.
    Does NOT drop anything — call retrain_pruned() once you've decided.

    Returns:
        importance — DataFrame with columns:
                     feature | mean_abs_shap | shap_pct | shap_cum_pct
    """
    print("\n── SHAP Feature Importance (Round 1) ───────────")

    sample      = X_train.sample(min(n_samples, len(X_train)), random_state=42)
    explainer   = shap.TreeExplainer(model)
    shap_values = explainer.shap_values(sample)

    mean_abs_shap = np.abs(shap_values).mean(axis=0)
    importance = (
        pd.DataFrame({"feature": feature_cols, "mean_abs_shap": mean_abs_shap})
        .sort_values("mean_abs_shap", ascending=False)
        .reset_index(drop=True)
    )
    importance["shap_pct"]     = 100 * importance["mean_abs_shap"] / importance["mean_abs_shap"].sum()
    importance["shap_cum_pct"] = importance["shap_pct"].cumsum()

    # Print table so you can read it in the console
    print()
    print(importance.to_string(index=False, float_format="{:.4f}".format))
    print()
    print("  ↳ Review the table and chart, then call retrain_pruned(drop_cols=[...]) "
          "with whichever features you want to remove.")

    # Bar chart — no threshold line, no colour coding; purely informational
    fig, ax = plt.subplots(figsize=(10, 0.45 * len(feature_cols) + 1.5))
    ax.barh(importance["feature"][::-1], importance["mean_abs_shap"][::-1], color="#2563eb")
    ax.set_xlabel("Mean |SHAP value|")
    ax.set_title("Round 1 — SHAP Feature Importance\n(inspect, then choose which to drop)")
    ax.grid(axis="x", alpha=0.3)
    plt.tight_layout()
    plt.savefig("/content/drive/MyDrive/LOS/shap_feature_importance.png", dpi=150)
    plt.close()
    print("  Saved: shap_feature_importance.png")

    return importance


def retrain_pruned(
    drop_cols: list,
    X_train: pd.DataFrame,
    X_val: pd.DataFrame,
    X_test: pd.DataFrame,
    y_train: pd.Series,
    y_val: pd.Series,
    y_raw_test: pd.Series,
    feature_cols: list,
    metrics_r1: dict,
) -> tuple:
    """
    Retrain (Round 2) after you've manually chosen which features to drop.

    Parameters
    ----------
    drop_cols   : list of feature names you want to remove, e.g.
                  ["ethnicity", "zip3", "gender"]
    ...rest     : pass the same split objects returned by split_data()
    metrics_r1  : the dict returned by compute_metrics() on the Round 1 model

    Returns
    -------
    model_r2, quantile_models, keep_cols
    """
    keep_cols = [c for c in feature_cols if c not in drop_cols]
    cat_cols_pruned = [c for c in CATEGORICAL_COLS if c in keep_cols]

    if not keep_cols:
        raise ValueError("No features left after dropping — check your drop_cols list.")

    print(f"\n══ ROUND 2 — Pruned Feature Set ════════════════")
    print(f"  Dropping: {drop_cols}")
    print(f"  Keeping:  {keep_cols}")

    model_r2 = train_median_model(
        X_train[keep_cols], y_train,
        X_val[keep_cols],   y_val,
        cat_cols_pruned,
    )
    metrics_r2 = compute_metrics(model_r2, X_test[keep_cols], y_raw_test)
    print(f"  MAE={metrics_r2['MAE']:.3f}  RMSE={metrics_r2['RMSE']:.3f}  "
          f"MdAE={metrics_r2['MdAE']:.3f}  n_features={metrics_r2['n_features']}")

    compare_rounds(metrics_r1, metrics_r2)

    print("\n══ Quantile Models (Round 2 features) ══════════")
    quantile_models = train_quantile_models(
        X_train[keep_cols], y_train,
        X_val[keep_cols],   y_val,
        cat_cols_pruned,
    )

    evaluate(model_r2, quantile_models, X_test[keep_cols], y_raw_test, keep_cols)
    explain(model_r2, X_test[keep_cols], keep_cols)

    model_r2.booster_.save_model("/content/drive/MyDrive/LOS/los_lgbm_model.txt")
    print("\n  Saved: los_lgbm_model.txt")

    # ── Save category mappings for inference ─────────────────────────────────
    # Persists the category levels seen during training so predict_single()
    # can align new data to the same encoding at inference time.
    # No OrdinalEncoder needed — LightGBM uses pandas category dtype directly.
    cat_cols_final = [c for c in CATEGORICAL_COLS if c in keep_cols]
    if cat_cols_final:
        category_maps = {
            col: X_train[col].cat.categories.tolist()
            for col in cat_cols_final
        }
        joblib.dump(
            {"category_maps": category_maps, "cat_cols": cat_cols_final, "feature_order": keep_cols},
            "/content/drive/MyDrive/LOS/los_encoder.pkl"
        )
        print("  Saved: los_encoder.pkl")
        print(f"  Encoded columns: {cat_cols_final}")

    print(f"\n✓ Round 2 complete. Final feature set: {keep_cols}")
    return model_r2, quantile_models, keep_cols


# ─────────────────────────────────────────────
# 8. COMPARE ROUND 1 vs ROUND 2
# ─────────────────────────────────────────────

def compute_metrics(model, X_test: pd.DataFrame, y_raw_test: pd.Series) -> dict:
    """Return MAE, RMSE, MdAE on original LOS scale."""
    preds  = np.exp(model.predict(X_test))
    y_true = y_raw_test.values
    return {
        "MAE":  mean_absolute_error(y_true, preds),
        "RMSE": np.sqrt(mean_squared_error(y_true, preds)),
        "MdAE": float(np.median(np.abs(y_true - preds))),
        "n_features": X_test.shape[1],
    }


def compare_rounds(metrics_r1: dict, metrics_r2: dict) -> None:
    """
    Print a side-by-side table of Round 1 vs Round 2 metrics.
    A good pruning keeps accuracy stable (< ~2% MAE change) while
    reducing feature count.
    """
    print("\n── Round 1 vs Round 2 Comparison ──────────────")
    header = f"{'Metric':<12} {'Round 1':>10} {'Round 2':>10} {'Δ':>10}"
    print(header)
    print("─" * len(header))

    for key in ["MAE", "RMSE", "MdAE"]:
        r1, r2 = metrics_r1[key], metrics_r2[key]
        delta  = r2 - r1
        flag   = "✓" if abs(delta / r1) < 0.02 else "⚠ >2%"
        print(f"  {key:<10} {r1:>10.3f} {r2:>10.3f} {delta:>+10.3f}  {flag}")

    print(f"  {'Features':<10} {metrics_r1['n_features']:>10} "
          f"{metrics_r2['n_features']:>10} "
          f"{metrics_r2['n_features'] - metrics_r1['n_features']:>+10}")

    # Bar chart comparison
    metrics_to_plot = ["MAE", "RMSE", "MdAE"]
    r1_vals = [metrics_r1[m] for m in metrics_to_plot]
    r2_vals = [metrics_r2[m] for m in metrics_to_plot]

    x = np.arange(len(metrics_to_plot))
    width = 0.35
    fig, ax = plt.subplots(figsize=(8, 4))
    ax.bar(x - width/2, r1_vals, width, label=f"Round 1 ({metrics_r1['n_features']} features)",
           color="#93c5fd")
    ax.bar(x + width/2, r2_vals, width, label=f"Round 2 ({metrics_r2['n_features']} features)",
           color="#2563eb")
    ax.set_xticks(x)
    ax.set_xticklabels(metrics_to_plot)
    ax.set_ylabel("Days")
    ax.set_title("Round 1 vs Round 2 — Accuracy After Feature Pruning")
    ax.legend()
    ax.grid(axis="y", alpha=0.3)
    for i, (v1, v2) in enumerate(zip(r1_vals, r2_vals)):
        delta = v2 - v1
        ax.text(i + width/2, v2 + 0.01, f"{delta:+.3f}", ha="center",
                fontsize=8, color="#6b7280")
    plt.tight_layout()
    plt.savefig("/content/drive/MyDrive/LOS/round_comparison.png", dpi=150)
    plt.close()
    print("  Saved: round_comparison.png")


# ─────────────────────────────────────────────
# 9. MAIN PIPELINE
# ─────────────────────────────────────────────

def main(filepath: str):
    """
    Runs Round 1 only — trains on all features, prints SHAP importance,
    and saves a chart for you to inspect.

    After reviewing, call retrain_pruned() with the features you want to drop:

        model, q_models, final_features = retrain_pruned(
            drop_cols=["ethnicity", "zip3"],
            X_train=X_train, X_val=X_val, X_test=X_test,
            y_train=y_train, y_val=y_val, y_raw_test=y_raw_test,
            feature_cols=feature_cols,
            metrics_r1=metrics_r1,
        )
    """
    # ── Load & prepare ────────────────────────────────────────────────────────
    df = load_sparcs(filepath)
    df = clean(df)
    df, feature_cols = engineer_features(df)
    cat_cols = [c for c in CATEGORICAL_COLS if c in feature_cols]

    (X_train, X_val, X_test,
     y_train, y_val, y_test,
     y_raw_train, y_raw_val, y_raw_test) = split_data(df, feature_cols)

    # ── ROUND 1: train on all features ───────────────────────────────────────
    print("\n══ ROUND 1 — Full Feature Set ══════════════════")
    model_r1   = train_median_model(X_train, y_train, X_val, y_val, cat_cols)
    metrics_r1 = compute_metrics(model_r1, X_test[feature_cols], y_raw_test)
    print(f"  MAE={metrics_r1['MAE']:.3f}  RMSE={metrics_r1['RMSE']:.3f}  "
          f"MdAE={metrics_r1['MdAE']:.3f}  n_features={metrics_r1['n_features']}")

    # ── SHAP importance — inspect, don't drop yet ─────────────────────────────
    importance = compute_shap_importance(model_r1, X_train, feature_cols)

    # Return everything so you can pass it to retrain_pruned() interactively
    return dict(
        model_r1     = model_r1,
        metrics_r1   = metrics_r1,
        importance   = importance,
        feature_cols = feature_cols,
        X_train=X_train, X_val=X_val, X_test=X_test,
        y_train=y_train, y_val=y_val,
        y_raw_test=y_raw_test,
    )

## First Model Training

In [ ]:
# ─────────────────────────────────────────────
# USAGE  (run in a notebook or interactive Python session)
# ─────────────────────────────────────────────

if __name__ == "__main__":
    DATA_PATH = "/content/drive/MyDrive/LOS/Patient_CSV_NYC.csv"

    # ── STEP 1: Train Round 1, view SHAP importance ───────────────────────────
    state = main(DATA_PATH)

    # Inspect state["importance"] and shap_feature_importance.png, then…

    # ── STEP 2: Drop whichever features you choose, retrain, compare ──────────
    # model, q_models, final_features = retrain_pruned(
    #     drop_cols=["ethnicity","med_surg"],   # ← your call after inspecting SHAP
    #     X_train    = state["X_train"],
    #     X_val      = state["X_val"],
    #     X_test     = state["X_test"],
    #     y_train    = state["y_train"],
    #     y_val      = state["y_val"],
    #     y_raw_test = state["y_raw_test"],
    #     feature_cols = state["feature_cols"],
    #     metrics_r1   = state["metrics_r1"],
    # )ethnicity: The least important feature (nearly zero impact).



# ─────────────────────────────────────────────
# QUICK INFERENCE EXAMPLE
# ─────────────────────────────────────────────

def predict_single(model, q_models, patient_dict: dict, feature_cols: list) -> dict:
    """
    Predict LOS for a single patient at admission.
    
    Example:
        patient = {
            "age_group": "70 or Older",
            "gender": "M",
            "race": "White",
            "ethnicity": "Not Span/Hispanic",
            "admission_type": "Emergency",
            "ccs_dx": 108,          # Congestive heart failure
            "ccs_proc": 216,
            "apr_drg": 194,
            "apr_severity": 3,
            "apr_rom": 3,
            "med_surg": "Medical",
            "health_service_area": "New York City",
            "zip3": "100",
        }
    """
    row = pd.DataFrame([patient_dict])
    for col in feature_cols:
        if col not in row.columns:
            row[col] = np.nan
    row = row[feature_cols]

    log_pred = model.predict(row)[0]
    pred_days = np.exp(log_pred)

    result = {
        "predicted_los_days": round(pred_days, 1),
        "predicted_los_rounded": max(1, round(pred_days)),
    }

    if q_models:
        result["interval_10th"] = round(np.exp(q_models[0.10].predict(row)[0]), 1)
        result["interval_25th"] = round(np.exp(q_models[0.25].predict(row)[0]), 1)
        result["interval_75th"] = round(np.exp(q_models[0.75].predict(row)[0]), 1)
        result["interval_90th"] = round(np.exp(q_models[0.90].predict(row)[0]), 1)

    return result

## Prune Model - Removing Features

In [ ]:

    # ── STEP 2: Drop whichever features you choose, retrain, compare ──────────
model, q_models, final_features = retrain_pruned(
    drop_cols=["ethnicity","med_surg","gender","race","health_service_area","zip3"],   # ← your call after inspecting SHAP
    X_train    = state["X_train"],
    X_val      = state["X_val"],
    X_test     = state["X_test"],
    y_train    = state["y_train"],
    y_val      = state["y_val"],
    y_raw_test = state["y_raw_test"],
    feature_cols = state["feature_cols"],
    metrics_r1   = state["metrics_r1"])